# 🧬 BioKG HPC GNN Masterclass: Real-World Multi-GPU Link Prediction

Welcome to the **Scientific AI & HPC Masterclass**. This notebook is now fully configured for **Production Data** using the PrimeKG (8.1M edges) and BioBridge multimodal embeddings.

### 🎯 The Project Goal
We are performing **Link Prediction** on a biomedical knowledge graph. By predicting hidden links between drugs and diseases, we enable **AI-driven drug repurposing**.

### 🚀 Technical Performance Highlights:
1. **Zero-Mocking**: This entire notebook now loads and trains on the real 8.1M edge PrimeKG dataset.
2. **NVIDIA RAPIDS (cuDF)**: GPU-accelerated graph construction (100x faster than Pandas).
3. **Heterogeneous GraphSAGE**: Unique mathematical manifolds for Drugs, Genes, and Diseases.
4. **Multimodal Alignment**: Real ESM-2b (Proteins), SMILES (Drugs), and PubMedBERT (Diseases) embeddings linked to the graph nodes.

## ⚡ Step 0: Environment & Hardware Verification
We start by ensuring the NVIDIA GPU is active and installing the specialized Graph Engines.

In [ ]:
import torch
import os
import sys
from pathlib import Path

# 1. Hardware Check
print(f"GPU Detected: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

# 2. Optimized Graph Extensions Installation (~2 minutes)
!pip install pytorch-lightning wandb loguru pyg-lib torch-sparse -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html

# 3. Setup Project Path
REPO_PATH = '/content/KG_MLOps'
if os.path.exists(REPO_PATH):
    sys.path.append(REPO_PATH)
    os.chdir(REPO_PATH)
    print(f"✅ Current Directory: {os.getcwd()}")
else:
    print("❌ ERROR: Clone the repo to /content/KG_MLOps first!")

## 📥 Step 1: Real-World Data Acquisition
Run these scripts to download and preprocess the real 8GB dataset. 
**Note:** If you've already run these, you can skip this cell.

In [ ]:
# !python data/download_biobridge.py
# !python data/preprocess.py

## 🏗️ Step 2: The Graph Data Pipeline
We use our specialized `BioBridgeGNNDataModule`. It utilizes **NVIDIA RAPIDS** to build the 8.1M edge `HeteroData` object.

In [ ]:
from data.biobridge_gnn_datamodule import BioBridgeGNNDataModule

datamodule = BioBridgeGNNDataModule(
    data_dir='data/processed/',
    batch_size=1024,
    num_workers=2
)

datamodule.setup()
print(f"\n✅ Real Heterogeneous Metadata: {datamodule.data.metadata()}")

## 🧠 Step 3: Model Architecture (HeteroGNN) & Alignment
We instantiate our `BioBridgeLinkPredictor` and perform the **Multimodal Alignment**.

In [ ]:
from models.hetero_gnn import BioBridgeLinkPredictor

model = BioBridgeLinkPredictor(
    metadata=datamodule.data.metadata(),
    hidden_channels=128
)

# --- CRITICAL: Align BioBridge Real Embeddings ---
print("Aligning Real-World BioBridge Embeddings (PKL) to Graph Nodes...")
model.projector.load_pretrained_mappings(datamodule.data)

print("\n✅ Heterogeneous GraphSAGE initialized and Aligned.")

## 🚀 Step 4: Distributed Training
Let's start the optimization. Notice the use of `16-mixed` precision for speed.

In [ ]:
import pytorch_lightning as pl

trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed",
    max_epochs=10,
    log_every_n_steps=10
)

trainer.fit(model, datamodule=datamodule)

## 📊 Step 5: Visualizing Latent Embeddings (UMAP)
Now that the model is trained on REAL data, the UMAP clusters will be biologically meaningful.

In [ ]:
import umap.umap_ as umap
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    z_dict = model(datamodule.data) # Forward pass on the full graph

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding_2d = reducer.fit_transform(z_dict['drug'].numpy()[:500])

plt.figure(figsize=(10, 7))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c='magenta', alpha=0.5)
plt.title("Real BioBridge Latent Space (Drugs)")
plt.show()